In [1]:
import sys
sys.path.append("..")


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from factor_engines.load_data import get_price_data, compute_returns
from factor_engines.factors import (
    momentum_factor,
    volatility_factor,
    reversal_factor,
    sma_distance_factor,
    compute_all,
    zscore,
    daily_score
)
from factor_engines.portfolio import (
    build_long_short_portfolio,
    build_zscore_portfolio_from_factor,
    build_composite_portfolio,
    composite_factor
)
from factor_engines.regression import (
    summarize_regression
)

plt.style.use("seaborn-v0_8")


In [3]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

prices = get_price_data(tickers, "2015-01-01", "2025-01-01")
returns = compute_returns(prices["Close"])




In [4]:
momentum = momentum_factor(prices)
volatility = volatility_factor(prices)
reversal = reversal_factor(prices)
sma = sma_distance_factor(prices)

factors = {
    "momentum": momentum,
    "volatility": volatility,
    "reversal": reversal,
    "sma_distance": sma
}


In [5]:
mom_weights_ls, mom_ret_ls = build_long_short_portfolio(momentum, returns)
mom_weights_z, mom_ret_z = build_zscore_portfolio_from_factor(momentum, returns)
comp_weights, comp_ret = build_composite_portfolio(factors, returns)


In [6]:
market_prices = get_price_data(["SPY"], "2015-01-01", "2025-01-01")
close_prices = market_prices["Close"]
spy_prices = close_prices["SPY"]
market_ret = compute_returns(spy_prices)

market_ret.head()


Date
2015-01-05   -0.018060
2015-01-06   -0.009419
2015-01-07    0.012461
2015-01-08    0.017745
2015-01-09   -0.008014
Name: SPY, dtype: float64

In [7]:
comp_weights.sum(axis=1).head(30)


Date
2016-01-04   -1.040834e-16
2016-01-05   -5.605442e-02
2016-01-06   -1.747061e-02
2016-01-07   -3.267819e-02
2016-01-08   -6.938894e-18
2016-01-11   -1.387779e-17
2016-01-12   -5.080119e-02
2016-01-13    1.318390e-16
2016-01-14    2.775558e-17
2016-01-15   -4.163336e-17
2016-01-19   -6.938894e-18
2016-01-20    2.775558e-17
2016-01-21    0.000000e+00
2016-01-22    1.110223e-16
2016-01-25   -1.214012e-02
2016-01-26   -2.081668e-17
2016-01-27    5.551115e-17
2016-01-28   -4.861917e-02
2016-01-29   -9.442398e-03
2016-02-01   -1.387779e-17
2016-02-02    1.523350e-02
2016-02-03   -3.913970e-17
2016-02-04    1.240327e-16
2016-02-05   -1.784011e-02
2016-02-08   -5.343808e-03
2016-02-09   -2.550365e-03
2016-02-10   -1.867637e-02
2016-02-11   -1.734723e-17
2016-02-12   -2.974289e-03
2016-02-16   -6.005798e-02
dtype: float64

In [8]:
combined = pd.concat(
    [comp_ret, market_ret],
    axis=1,
    keys=["comp", "SPY"],
    join="inner"
)

combined.head(20), combined.shape


(                comp       SPY
 Date                          
 2016-01-04 -0.015237 -0.013980
 2016-01-05  0.003548  0.001691
 2016-01-06  0.006088 -0.012614
 2016-01-07  0.003087 -0.023991
 2016-01-08 -0.001565 -0.010977
 2016-01-11  0.001190  0.000990
 2016-01-12 -0.005072  0.008069
 2016-01-13 -0.010864 -0.024941
 2016-01-14 -0.001202  0.016416
 2016-01-15 -0.002933 -0.021466
 2016-01-19  0.005699  0.001331
 2016-01-20 -0.002110 -0.012815
 2016-01-21  0.001539  0.005602
 2016-01-22 -0.000812  0.020516
 2016-01-25  0.002184 -0.015117
 2016-01-26  0.000106  0.013643
 2016-01-27  0.007691 -0.010883
 2016-01-28 -0.002042  0.005209
 2016-01-29 -0.048137  0.024377
 2016-02-01 -0.000227 -0.000361,
 (2264, 2))

In [9]:
from factor_engines.regression import summarize_regression

summary_comp, results_comp = summarize_regression(comp_ret, market_ret)
summary_comp


{'alpha_daily': np.float64(0.0001038059486326269),
 'alpha_annual': np.float64(0.02615909905542198),
 'alpha_tstat': np.float64(1.0981752083108665),
 'r2': np.float64(0.003733644011636672),
 'beta_SPY': 0.024455526548459207,
 'beta_SPY_tstats': 2.9115551561755435}